In [1]:
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack

In [3]:
df_movies = pd.read_csv('/content/netflix_cleaned.csv')

In [4]:
print("5 baris pertama dataset:")
display(df_movies.head())

5 baris pertama dataset:


,Title,parent_genre_str,metadata,poster,year,IMDB Score,Runtime,runtime_normalized,primary_language,Premiere
0,enter the anime,documentary,movie title is enter the anime. genre is docum...,https://m.media-amazon.com/images/M/MV5BNzljM2...,2019.0,2.5,58,0.263415,english,"August 5, 2019"
1,dark forces,thriller,movie title is dark forces. genre is thriller....,https://m.media-amazon.com/images/M/MV5BOTIxMW...,2020.0,2.6,81,0.375610,spanish,"August 21, 2020"
2,the app,"scifi, drama","movie title is the app. genre is scifi, drama....",https://m.media-amazon.com/images/M/MV5BNzgzZG...,2019.0,2.6,79,0.365854,italian,"December 26, 2019"
3,the open house,"horror, thriller",movie title is the open house. genre is horror...,https://m.media-amazon.com/images/M/MV5BMTU0Mj...,2018.0,3.2,94,0.439024,english,"January 19, 2018"
4,kaali khuhi,mystery,movie title is kaali khuhi. genre is mystery. ...,https://m.media-amazon.com/images/M/MV5BZjRhNG...,2020.0,3.4,90,0.419512,hindi,"October 30, 2020"


In [5]:
num_unique_genres = df_movies['parent_genre_str'].nunique()
print(f"Jumlah kategori genre unik di dataset adalah: {num_unique_genres}")

Jumlah kategori genre unik di dataset adalah: 57


In [6]:
unique_genres = df_movies['parent_genre_str'].unique()
print("\nInformasi dataset:")
display(df_movies.info())


Informasi dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 529 entries, 0 to 528
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Title               529 non-null    object 
 1   parent_genre_str    529 non-null    object 
 2   metadata            529 non-null    object 
 3   poster              529 non-null    object 
 4   year                529 non-null    float64
 5   IMDB Score          529 non-null    float64
 6   Runtime             529 non-null    int64  
 7   runtime_normalized  529 non-null    float64
 8   primary_language    529 non-null    object 
 9   Premiere            529 non-null    object 
dtypes: float64(3), int64(1), object(6)
memory usage: 41.5+ KB


None

In [7]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(df_movies['parent_genre_str'])
print(f"Bentuk matriks TF-IDF untuk genre: {tfidf_matrix.shape}")

Bentuk matriks TF-IDF untuk genre: (529, 16)


In [8]:
scaler = MinMaxScaler()
imdb_normalized = scaler.fit_transform(df_movies[['IMDB Score']])

In [9]:
runtime_normalized = df_movies[['runtime_normalized']].values

In [11]:
from scipy.sparse import csc_matrix

imdb_sparse = csc_matrix(imdb_normalized)
runtime_sparse = csc_matrix(runtime_normalized)
numeric_features = hstack([imdb_sparse, runtime_sparse])
print(f"Bentuk matriks fitur numerik setelah penyelarasan: {numeric_features.shape}")

Bentuk matriks fitur numerik setelah penyelarasan: (529, 2)


In [12]:
feature_matrix = hstack([tfidf_matrix, numeric_features])
print(f"Bentuk matriks fitur akhir: {feature_matrix.shape}")

Bentuk matriks fitur akhir: (529, 18)


In [13]:
cosine_sim = cosine_similarity(feature_matrix)
print(f"Bentuk matriks cosine similarity: {cosine_sim.shape}")

Bentuk matriks cosine similarity: (529, 529)


In [14]:
joblib.dump(tfidf_vectorizer, 'api_tfidf_vectorizer.pkl')
print('api_tfidf_vectorizer.pkl berhasil disimpan.')

joblib.dump(feature_matrix, 'api_feature_matrix.pkl')
print('api_feature_matrix.pkl berhasil disimpan.')

kolom_relevan = ['Title', 'parent_genre_str', 'IMDB Score', 'Runtime', 'poster', 'primary_language', 'Premiere']
df_movies[kolom_relevan].to_pickle('api_df_movies.pkl')
print('api_df_movies.pkl berhasil disimpan.')

api_tfidf_vectorizer.pkl berhasil disimpan.
api_feature_matrix.pkl berhasil disimpan.
api_df_movies.pkl berhasil disimpan.


In [19]:
import pandas as pd
import joblib
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from scipy.sparse import csr_matrix

df_movies = pd.read_pickle('api_df_movies.pkl')
feature_matrix = joblib.load('api_feature_matrix.pkl')

feature_matrix = feature_matrix.tocsr()

def dapatkan_rekomendasi_demo(input_genres, top_n_per_genre=3):
    rekomendasi_akhir = []

    for genre in input_genres:
        kondisi_genre = df_movies['parent_genre_str'].str.contains(genre, case=False, na=False)

        if not kondisi_genre.any():
            continue

        indeks_film_genre = df_movies[kondisi_genre].index

        vektor_genre_rata_rata = feature_matrix[indeks_film_genre].mean(axis=0)

        if isinstance(vektor_genre_rata_rata, np.matrix):
            vektor_genre_rata_rata = np.asarray(vektor_genre_rata_rata)

        skor_kemiripan = cosine_similarity(vektor_genre_rata_rata, feature_matrix).flatten()

        indeks_terurut = np.argsort(skor_kemiripan)[::-1]

        hitung = 0
        for idx in indeks_terurut:
            if hitung >= top_n_per_genre:
                break
            film = df_movies.iloc[idx].to_dict()
            film['skor_kemiripan'] = skor_kemiripan[idx]
            film['sumber_genre_pop_up'] = genre

            if film['Title'] not in [f['Title'] for f in rekomendasi_akhir]:
                rekomendasi_akhir.append(film)
                hitung += 1

    return pd.DataFrame(rekomendasi_akhir)

# --- SIMULASI DEMO ---
genre_pilihan_user = ['thriller', 'comedy', 'documentary']

df_hasil_rekomendasi = dapatkan_rekomendasi_demo(genre_pilihan_user, top_n_per_genre=3)
display(df_hasil_rekomendasi)

,Title,parent_genre_str,IMDB Score,Runtime,poster,primary_language,Premiere,skor_kemiripan,sumber_genre_pop_up
0,in the shadow of the moon,thriller,6.2,115,https://m.media-amazon.com/images/M/MV5BYWM3Yz...,english,"September 27, 2019",0.992348,thriller
1,a fall from grace,thriller,5.9,120,https://m.media-amazon.com/images/M/MV5BOTdhOG...,english,"January 17, 2020",0.992321,thriller
2,time to hunt,thriller,6.3,134,https://m.media-amazon.com/images/M/MV5BZTI2OT...,korean,"April 23, 2020",0.991960,thriller
3,rose island,comedy,7.0,117,https://m.media-amazon.com/images/M/MV5BMDhlOD...,italian,"December 9, 2020",0.964703,comedy
4,ludo,comedy,7.6,149,https://m.media-amazon.com/images/M/MV5BYmI3ZD...,hindi,"November 12, 2020",0.964583,comedy
5,have you ever seen fireflies?,comedy,6.2,114,https://m.media-amazon.com/images/M/MV5BYzgzZD...,turkish,"April 9, 2021",0.961837,comedy
6,keith richards: under the influence,documentary,7.1,81,https://m.media-amazon.com/images/M/MV5BMTc2MD...,english,"September 18, 2015",0.999544,documentary
7,dance dreams: hot chocolate nutcracker,documentary,7.1,80,https://m.media-amazon.com/images/M/MV5BYzdhNm...,english,"November 27, 2020",0.999542,documentary
8,joshua: teenager vs. superpower,documentary,7.1,78,https://m.media-amazon.com/images/M/MV5BMzhmMm...,english,"May 26, 2017",0.999499,documentary
